In [1]:
import pandas as pd
import pytz
import datetime as dt
from sklearn.linear_model import LinearRegression
import joblib
from sklearn.preprocessing import StandardScaler

In [2]:
def preprocess_QuantAQ(file_path, start_date, end_date):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp_local', 'no2']].dropna(subset=['no2'])
    sensor.rename(columns={'timestamp_local':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.tz_localize(None)
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')
    sensor = sensor[(sensor['time'] >= start_date) & (sensor['time'] <= end_date)]

    sensor = sensor[::-1].reset_index(drop=True)
    # train only on first 3/4 of data
    # split_index = i.int(len(sensor) * 0.75)
    # sensor = sensorloc[:split_index]
    return sensor


In [3]:
# returns a model that predicts daily no2 from hourly sensor data
def train_hourly_model(sensor_collocation_data, h_regular):
    temp = sensor_collocation_data.groupby('dayhour').agg(
        shourlyNO2=('no2', lambda x: x.mean(skipna=True)),
    ).reset_index()

    merged = pd.merge(h_regular, temp, on='dayhour').dropna()
    feature_cols = ['shourlyNO2']
    target = merged['rhourlyNO2']

    return LinearRegression().fit(merged[feature_cols], target)


def train_daily_model(sensor_collocation_data, d_regular):
    temp = sensor_collocation_data.groupby('day').agg(
        sdailyNO2=('no2', lambda x: x.mean(skipna=True)),
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    merged = pd.merge(temp, d_regular, on='day').dropna().sort_values('day')
    feature_cols = ['sdailyNO2']
    target = merged['rdailyNO2']

    return LinearRegression().fit(merged[feature_cols], target)



In [4]:
quantAQ00589 = preprocess_QuantAQ("../../ShortenedData/MOD-00589-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00590 = preprocess_QuantAQ("../../ShortenedData/MOD-00590-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00591 = preprocess_QuantAQ("../../ShortenedData/MOD-00591-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00592 = preprocess_QuantAQ("../../ShortenedData/MOD-00592-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00593 = preprocess_QuantAQ("../../ShortenedData/MOD-00593-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00595 = preprocess_QuantAQ("../../ShortenedData/MOD-00595-RAW.csv", '2025-05-31', '2025-07-14')

In [5]:
# load and clean GAPA data
gapa = pd.read_csv('../NO2&PM2.5.csv', header=2)
gapa.columns = ['Date', 'PMHR_2', 'PMHR', 'PMHR10', '24H_PMHR10', 'PMHRC', '24H_PMHR']
gapa = gapa[['Date', 'PMHR10']]

gapa['time'] = pd.to_datetime(gapa['Date'], format='%d-%b-%Y %H:%M')
gapa['day'] = gapa['time'].dt.date
gapa['dayhour'] = gapa['time'].dt.strftime("%Y-%m-%d %H")
gapa.rename(columns={'PMHR10':'rno2'}, inplace=True)

gapa = gapa.sort_values('time').reset_index(drop=True).dropna(subset=['rno2'])

# train only on first 3/4 of data
# split_index = int(len(gapa) * 0.75)
# gapa = gapa.iloc[:split_index]

h_gapa = gapa.groupby('dayhour').agg(
    rhourlyNO2=('rno2', lambda x: x.mean(skipna=True)),
).reset_index()

d_gapa = gapa.groupby('day').agg(
    rdailyNO2=('rno2', lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)),
).reset_index()




FileNotFoundError: [Errno 2] No such file or directory: '../NO2&PM2.5.csv'

In [ ]:
hmodel_00589 = train_hourly_model(quantAQ00589, h_gapa)
dmodel_00589 = train_daily_model(quantAQ00589, d_gapa)
hmodel_00590 = train_hourly_model(quantAQ00590, h_gapa)
dmodel_00590 = train_daily_model(quantAQ00590, d_gapa)
hmodel_00591 = train_hourly_model(quantAQ00591, h_gapa)
dmodel_00591 = train_daily_model(quantAQ00591, d_gapa)
hmodel_00592 = train_hourly_model(quantAQ00592, h_gapa)
dmodel_00592 = train_daily_model(quantAQ00592, d_gapa)
hmodel_00593 = train_hourly_model(quantAQ00593, h_gapa)
dmodel_00593 = train_daily_model(quantAQ00593, d_gapa)
hmodel_00595 = train_hourly_model(quantAQ00595, h_gapa)
dmodel_00595 = train_daily_model(quantAQ00595, d_gapa)

In [ ]:
# save models
joblib.dump(hmodel_00589, '../models/hmodel_00589.joblib')
joblib.dump(dmodel_00589, '../models/dmodel_00589.joblib')
joblib.dump(hmodel_00590, '../models/hmodel_00590.joblib')
joblib.dump(dmodel_00590, '../models/dmodel_00590.joblib')
joblib.dump(hmodel_00591, '../models/hmodel_00591.joblib')
joblib.dump(dmodel_00591, '../models/dmodel_00591.joblib')
joblib.dump(hmodel_00592, '../models/hmodel_00592.joblib')
joblib.dump(dmodel_00592, '../models/dmodel_00592.joblib')
joblib.dump(hmodel_00593, '../models/hmodel_00593.joblib')
joblib.dump(dmodel_00593, '../models/dmodel_00593.joblib')
joblib.dump(hmodel_00595, '../models/hmodel_00595.joblib')
joblib.dump(dmodel_00595, '../models/dmodel_00595.joblib')


['../models/dmodel_00595.joblib']